# 03 Tokenizéry: trénink a analýza (úkoly 1.5–1.9)

1. Připraví tokenizéry z `configs/tokenizer/`. BPE a Unigram se natrénují na náhodném
   vzorku trénovacích oken (100 tisíc oken, ~2 min a ~7 GB RAM na tokenizér),
   nukleotidy a k-mery se jen uloží.
2. Spočítá jejich statistiky na vzorku validačních oken (chr20).
3. Vykreslí grafy do práce (PDF + PNG).

Předpoklad: hotový notebook 02. Stačí CPU runtime.

## 1. Nastavení

In [ ]:
WINDOW = 1024

# Hlavní tokenizace + kontrolní překrývající se 6-mery. Unigram (T6) je volitelný: přidej "unigram_4096".
TOKENIZERS = ["nucleotide", "kmer6_overlap", "kmer6", "bpe_600", "bpe_4096"]
RETRAIN = False      # True = natrénovat znovu, i když už tokenizér uložený je
N_SAMPLE = 2000      # počet validačních oken pro statistiky

## Prostředí

Funguje v Colabu (prohlížeč i VS Code) i lokálně. V Colabu připojí Google Drive,
stáhne kód z GitHubu a zkopíruje genom na rychlý lokální disk. Jinde se cesty
řídí proměnnou `DNALM_ROOT`. Všechny cesty jsou v objektu `P` (`P.fasta`,
`P.windows("train")`, `P.tokenizer("bpe_600")`, `P.figures`, ...).

Ve VS Code s Colab rozšířením: pokud se Drive nepřipojí, použij příkaz
`Colab: Mount Google Drive to Server...` a buňku spusť znovu.

In [ ]:
# Úvodní buňka (stejná ve všech notebookách, neměnit).
# Colab: stáhne aktuální kód z GitHubu. Lokálně (Mac, školní GPU): použije nainstalované repo.
import importlib.util, sys
if importlib.util.find_spec("google.colab"):
    !git clone -q https://github.com/Valasik0/dna-lm.git /content/dna-lm 2>/dev/null || git -C /content/dna-lm pull -q
    !git -C /content/dna-lm log --oneline -1
    if "/content/dna-lm/src" not in sys.path:
        sys.path.insert(0, "/content/dna-lm/src")

from dnalm.utils.colab import setup
P = setup(window=WINDOW, genome=True)

## 2. Příprava a trénink tokenizérů

In [ ]:
import time
from dnalm.tokenizers import load_tokenizer
from dnalm.tokenizers.train import prepare_tokenizer

P.tokenizers_dir.mkdir(parents=True, exist_ok=True)
tokenizers = {}
for name in TOKENIZERS:
    path = P.tokenizer(name)
    if (path / "tokenizer_config.json").exists() and not RETRAIN:
        tokenizers[name] = load_tokenizer(str(path))
        print(f"[{name}] načten z {path}")
    else:
        t0 = time.time()
        tokenizers[name] = prepare_tokenizer(
            str(P.configs / "tokenizer" / f"{name}.yaml"),
            str(P.fasta), str(P.windows("train")), str(P.tokenizers_dir),
        )
        print(f"[{name}] hotovo za {time.time() - t0:.0f} s")
for name, tok in tokenizers.items():
    print(f"{name:15s} slovník {tok.vocab_size:>6,} tokenů")

## 3. Vzorek validačních oken

In [ ]:
import numpy as np
from dnalm.data.genome import GenomeWindows

# vzorek validačních oken (chr20); raw = se zachovanými malými písmeny (repetice)
val_raw = GenomeWindows(str(P.fasta), str(P.windows("val")), uppercase=False)
rng = np.random.default_rng(0)
idx = np.sort(rng.choice(len(val_raw), size=min(N_SAMPLE, len(val_raw)), replace=False))
raw_seqs = [val_raw[int(i)] for i in idx]
seqs = [s.upper() for s in raw_seqs]
print(f"{len(seqs)} validačních oken po {len(seqs[0])} bp")

## 4. Souhrnné statistiky

- `tokens_per_kb`: kolik tokenů dá 1 000 bp (délka vstupu modelu, a tedy cena výpočtu)
- `nt_per_token`: průměrný počet bází na token
- `single_nt_share`: podíl tokenů o délce 1 bp
- `vocab_used_share`: jaká část slovníku se ve vzorku skutečně použije
- `tokens_changed_mean`: kolik tokenů změní jedna bodová mutace (stabilita tokenizace)
- `resegmented_nt_mean`: jak dlouhý úsek se kolem mutace přesegmentuje (bp)
- `len_in_repeats`, `len_outside_repeats`: průměrná délka tokenu v repeticích a mimo ně

In [ ]:
import pandas as pd
from dnalm.tokenizers.stats import mutation_stability, repeat_token_lengths, summary
from dnalm.utils.plotting import label_for

rows = []
for name, tok in tokenizers.items():
    r = summary(tok, seqs)
    r.update({k: v for k, v in mutation_stability(tok, seqs[:200]).items() if k != "tokenizer"})
    r.update({k: v for k, v in repeat_token_lengths(tok, raw_seqs).items() if k != "tokenizer"})
    rows.append(r)
stats = pd.DataFrame(rows).set_index("tokenizer")
P.results("tokenizers").mkdir(parents=True, exist_ok=True)
stats.to_csv(P.results("tokenizers") / "tokenizer_stats.csv")

view = stats[["vocab_size", "tokens_per_kb", "nt_per_token", "token_len_max", "single_nt_share",
              "vocab_used_share", "tokens_changed_mean", "resegmented_nt_mean",
              "len_in_repeats", "len_outside_repeats"]].rename(index=label_for)
view.round(3)

## 5. Grafy

In [ ]:
import matplotlib.pyplot as plt
from dnalm.utils.plotting import apply_style, color_for, label_for, save_figure
apply_style()

# Tokeny na 1 kb: jedna míra napříč tokenizacemi -> sloupcový graf, hodnota přímo u sloupce
order = stats.sort_values("tokens_per_kb", ascending=False).index
fig, ax = plt.subplots(figsize=(6, 3.2))
bars = ax.barh([label_for(n) for n in order], stats.loc[order, "tokens_per_kb"],
               color=[color_for(n) for n in order], height=0.6)
ax.bar_label(bars, labels=[f"{v:,.0f}".replace(",", " ") for v in stats.loc[order, "tokens_per_kb"]],
             padding=4, fontsize=9, color="#0b0b0b")
ax.invert_yaxis()
ax.set_xlabel("tokenů na 1 000 bp")
ax.set_title("Délka vstupu modelu podle tokenizace", loc="left")
ax.grid(axis="y", visible=False)
ax.set_xlim(0, stats["tokens_per_kb"].max() * 1.15)
save_figure(fig, "tokens_per_kb", str(P.figures))
plt.show()

In [ ]:
from dnalm.tokenizers.stats import token_length_distribution

# Rozdělení délek tokenů jen pro naučené tokenizace (u nukleotidů a k-merů je délka pevná).
# Osa jen do MAX_LEN bp, podíl delších tokenů je v legendě (dlouhé tokeny = repetice).
learned = [n for n in tokenizers if n.startswith(("bpe", "unigram"))]
MAX_LEN = 20
fig, ax = plt.subplots(figsize=(6, 3.4))
for name in learned:
    dist = token_length_distribution(tokenizers[name], seqs)
    total = sum(dist.values())
    xs = list(range(1, MAX_LEN + 1))
    tail = sum(c for length, c in dist.items() if length > MAX_LEN) / total
    ax.plot(xs, [dist.get(x, 0) / total for x in xs], marker="o", markersize=4,
            color=color_for(name), label=f"{label_for(name)}, delší než {MAX_LEN} bp: {tail:.1%}")
ax.set_xlabel("délka tokenu (bp)")
ax.set_ylabel("podíl výskytů tokenů")
ax.set_xticks(range(1, MAX_LEN + 1, 2))
ax.set_title("Rozdělení délek tokenů na validačních oknech", loc="left")
ax.legend(fontsize=8.5)
save_figure(fig, "token_length_distribution", str(P.figures))
plt.show()

## 6. Co se naučené tokenizace naučily

In [ ]:
from dnalm.tokenizers.stats import top_tokens

for name in learned:
    tok = tokenizers[name]
    top = top_tokens(tok, seqs, n=15)
    longest = sorted((t for t in tok.vocab if not t.startswith("[")), key=len, reverse=True)[:10]
    print(f"=== {label_for(name)}")
    print("nejčastější:", ", ".join(f"{t} ({c})" for t, c in top))
    print("nejdelší:   ", ", ".join(t if len(t) <= 40 else t[:37] + "..." for t in longest))

## 7. Ukázka tokenizace jednoho úseku

In [ ]:
# Stejný úsek rozdělený všemi tokenizacemi (podklad pro obrázek v kapitole Tokenizace)
example = seqs[0][:60]
print(f"{'sekvence':26s} {example}")
for name, tok in tokenizers.items():
    print(f"{label_for(name):26s} {' | '.join(tok.tokenize(example))}")